In [1]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass

test = TestClass()
test.test()          # IT'S WORKING!

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 185 (delta 9), reused 21 (delta 6), pack-reused 159 (from 1)
Receiving objects: 100% (185/185), 81.80 MiB | 31.24 MiB/s, done.
Resolving deltas: 100% (92/92), done.
Github classes initialized!


In [2]:
from pathlib import Path

import cupy as cp
import numpy as np

from conv2d import Conv2D
from batchnorm2d import BatchNorm2D
from global_avg_pool2d import GlobalAvgPool2D
from leaky_relu import LeakyReLU
from linear import Linear
from maxpool import MaxPool2D
from flatten import Flatten
from dropout import Dropout


class YOLO:
    """
    YOLOv1

    Input:  (N, 3, 448, 448)
    Output: (N, 7, 7, 30) logits
    """

    def __init__(
        self,
        num_classes: int = 1000,
        dtype=cp.float32,
        negative_slope: float = 0.1,
    ):
        self.dtype = dtype
        self.num_classes = num_classes

        self.backbone = [
          Conv2D(3, 64, kernel_size=7, stride=2, padding=3, bias=False, dtype=dtype),
          BatchNorm2D(64, dtype=dtype),
          LeakyReLU(negative_slope),
          MaxPool2D(2, 2),

          Conv2D(64, 192, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(192, dtype=dtype),
          LeakyReLU(negative_slope),
          MaxPool2D(2, 2),

          Conv2D(192, 128, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(128, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(128, 256, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(256, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(256, 256, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(256, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(256, 512, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),
          MaxPool2D(2, 2),

          Conv2D(512, 256, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(256, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(256, 512, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(512, 256, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(256, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(256, 512, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(512, 256, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(256, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(256, 512, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(512, 256, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(256, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(256, 512, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(512, 512, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(512, 1024, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(1024, dtype=dtype),
          LeakyReLU(negative_slope),
          MaxPool2D(2, 2),

          Conv2D(1024, 512, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(512, 1024, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(1024, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(1024, 512, kernel_size=1, stride=1, padding=0, bias=False, dtype=dtype),
          BatchNorm2D(512, dtype=dtype),
          LeakyReLU(negative_slope),

          Conv2D(512, 1024, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
          BatchNorm2D(1024, dtype=dtype),
          LeakyReLU(negative_slope),
        ]

        self.head = [
            Conv2D(1024, 1024, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
            BatchNorm2D(1024, dtype=dtype),
            LeakyReLU(negative_slope),

            Conv2D(1024, 1024, kernel_size=3, stride=2, padding=1, bias=False, dtype=dtype),
            BatchNorm2D(1024, dtype=dtype),
            LeakyReLU(negative_slope),

            Conv2D(1024, 1024, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
            BatchNorm2D(1024, dtype=dtype),
            LeakyReLU(negative_slope),

            Conv2D(1024, 1024, kernel_size=3, stride=1, padding=1, bias=False, dtype=dtype),
            BatchNorm2D(1024, dtype=dtype),
            LeakyReLU(negative_slope),

            Flatten(),

            Linear(50176, 4096, bias=False),
            Dropout(p=0.5),
            LeakyReLU(negative_slope),

            Linear(4096, 7 * 7 * 30, bias=False),
        ]

    def forward(self, x: cp.ndarray) -> cp.ndarray:
        assert x.ndim == 4, f"Expected NCHW, got shape {x.shape}"
        assert x.shape[1] == 3, f"Expected 3 input channels, got {x.shape[1]}"
        for layer in self.backbone:
            x = layer.forward(x)
        for layer in self.head:
            print(type(layer), x.shape)
            x = layer.forward(x)
        return x

    def backward(self, grad_logits: cp.ndarray) -> cp.ndarray:
        assert grad_logits.ndim == 2, f"Expected 2D logits grad, got {grad_logits.shape}"
        final_out_features = self.head[-1].out_features
        assert grad_logits.shape[1] == final_out_features, (
            f"Expected (*, {final_out_features}), got {grad_logits.shape}"
        )

        grad = grad_logits
        for layer in reversed(self.head):
            grad = layer.backward(grad)
        for layer in reversed(self.backbone):
            grad = layer.backward(grad)

        return grad

    def _zero_grad_helper(self, layer):
        if isinstance(layer, Conv2D):
            layer.dW = cp.zeros_like(layer.weights)
            if layer.bias is not None:
                layer.db = cp.zeros_like(layer.bias)

        elif isinstance(layer, BatchNorm2D):
            if layer.affine:
                layer.dgamma = cp.zeros_like(layer.gamma)
                layer.dbeta = cp.zeros_like(layer.beta)

        elif isinstance(layer, Linear):
            layer.dW = cp.zeros_like(layer.W)
            if layer.use_bias:
                layer.db = cp.zeros_like(layer.b)

        elif isinstance(layer, Flatten):
            pass

        elif isinstance(layer, Dropout):
            pass

        elif isinstance(layer, LeakyReLU):
            pass

        elif isinstance(layer, MaxPool2D):
            pass

        else:
            raise NotImplementedError("Unsupported layer")

    def zero_grad(self):
        for layer in self.backbone:
            self._zero_grad_helper(layer)
        for layer in self.head:
            self._zero_grad_helper(layer)

    def _sgd_step_helper(self, layer, lr):
        if isinstance(layer, Conv2D):
            layer.weights -= lr * layer.dW
            if layer.bias is not None:
                layer.bias -= lr * layer.db

        elif isinstance(layer, BatchNorm2D):
            if layer.affine:
                layer.gamma -= lr * layer.dgamma
                layer.beta -= lr * layer.dbeta

        elif isinstance(layer, Linear):
            layer.W -= lr * layer.dW
            if layer.use_bias:
                layer.b -= lr * layer.db

        elif isinstance(layer, Flatten):
            pass

        elif isinstance(layer, Dropout):
            pass

        elif isinstance(layer, LeakyReLU):
            pass

        elif isinstance(layer, MaxPool2D):
            pass

        else:
            raise NotImplementedError("Unsupported layer")

    def sgd_step(self, learning_rate: float):
        lr = float(learning_rate)
        for layer in self.backbone:
            self._sgd_step_helper(layer, lr)
        for layer in self.head:
            self._sgd_step_helper(layer, lr)


    def save_weights(self, path: str | Path) -> None:
        path = Path(path)
        payload = {}
        conv_idx = 0
        bn_idx = 0
        fc_idx = 0
        def save_layer(layer):
            nonlocal conv_idx, bn_idx, fc_idx

            if isinstance(layer, Conv2D):
                payload[f"conv{conv_idx}_weights"] = cp.asnumpy(layer.weights)
                if layer.bias is not None:
                    payload[f"conv{conv_idx}_bias"] = cp.asnumpy(layer.bias)
                conv_idx += 1
            elif isinstance(layer, BatchNorm2D):
                if layer.affine:
                    payload[f"bn{bn_idx}_gamma"] = cp.asnumpy(layer.gamma)
                    payload[f"bn{bn_idx}_beta"] = cp.asnumpy(layer.beta)
                payload[f"bn{bn_idx}_running_mean"] = cp.asnumpy(layer.running_mean)
                payload[f"bn{bn_idx}_running_var"] = cp.asnumpy(layer.running_var)
                bn_idx += 1
            elif isinstance(layer, Linear):
                payload[f"fc{fc_idx}_W"] = cp.asnumpy(layer.W)
                if layer.use_bias:
                    payload[f"fc{fc_idx}_b"] = cp.asnumpy(layer.b)
                fc_idx += 1

        for layer in self.backbone:
            save_layer(layer)
        for layer in self.head:
            save_layer(layer)

        np.savez_compressed(path, **payload)

    def load_weights(self, path: str | Path) -> None:
        path = Path(path)
        data = np.load(path, allow_pickle=False)

        conv_idx = 0
        bn_idx = 0
        fc_idx = 0
        def load_layer(layer):
            nonlocal conv_idx, bn_idx, fc_idx
            if isinstance(layer, Conv2D):
                layer.weights = cp.asarray(data[f"conv{conv_idx}_weights"], dtype=self.dtype)

                key_b = f"conv{conv_idx}_bias"
                if layer.bias is not None:
                    if key_b not in data.files:
                        raise KeyError(f"Missing {key_b} in checkpoint")
                    layer.bias = cp.asarray(data[key_b], dtype=self.dtype)

                conv_idx += 1

            elif isinstance(layer, BatchNorm2D):
                if layer.affine:
                    layer.gamma = cp.asarray(data[f"bn{bn_idx}_gamma"], dtype=self.dtype)
                    layer.beta = cp.asarray(data[f"bn{bn_idx}_beta"], dtype=self.dtype)

                layer.running_mean = cp.asarray(data[f"bn{bn_idx}_running_mean"], dtype=self.dtype)
                layer.running_var = cp.asarray(data[f"bn{bn_idx}_running_var"], dtype=self.dtype)

                bn_idx += 1
            elif isinstance(layer, Linear):
                layer.W = cp.asarray(data[f"fc{fc_idx}_W"], dtype=self.dtype)
                if layer.use_bias:
                    layer.b = cp.asarray(data[f"fc{fc_idx}_b"], dtype=self.dtype)
                fc_idx += 1

        for layer in self.backbone:
            load_layer(layer)
        for layer in self.head:
            load_layer(layer)


    def __call__(self, x: cp.ndarray) -> cp.ndarray:
        return self.forward(x)

In [3]:
from darknet import Darknet

DARKNET_PATH = "/content/yolov1-cupy/models/darknet_pretrained-epoch15.npz"
backbone = Darknet(num_classes=10)
backbone.load_weights(DARKNET_PATH)

yolo = YOLO()

for i, b_layer in enumerate(backbone.layers):
    yolo.backbone[i] = b_layer

yolo.save_weights("/content/yolo-backbone-only.npz")

In [4]:
yolo = YOLO()
yolo.load_weights("/content/yolo-backbone-only.npz")

x = cp.random.rand(8, 3, 448, 448)
y = yolo(x)

print(y.shape)

<class 'conv2d.Conv2D'> (8, 1024, 14, 14)
<class 'batchnorm2d.BatchNorm2D'> (8, 1024, 14, 14)
<class 'leaky_relu.LeakyReLU'> (8, 1024, 14, 14)
<class 'conv2d.Conv2D'> (8, 1024, 14, 14)
<class 'batchnorm2d.BatchNorm2D'> (8, 1024, 7, 7)
<class 'leaky_relu.LeakyReLU'> (8, 1024, 7, 7)
<class 'conv2d.Conv2D'> (8, 1024, 7, 7)
<class 'batchnorm2d.BatchNorm2D'> (8, 1024, 7, 7)
<class 'leaky_relu.LeakyReLU'> (8, 1024, 7, 7)
<class 'conv2d.Conv2D'> (8, 1024, 7, 7)
<class 'batchnorm2d.BatchNorm2D'> (8, 1024, 7, 7)
<class 'leaky_relu.LeakyReLU'> (8, 1024, 7, 7)
<class 'flatten.Flatten'> (8, 1024, 7, 7)
<class 'linear.Linear'> (8, 50176)
<class 'dropout.Dropout'> (8, 4096)
<class 'leaky_relu.LeakyReLU'> (8, 4096)
<class 'linear.Linear'> (8, 4096)
(8, 1470)


In [5]:
yolo = YOLO()
yolo.load_weights("/content/yolo-backbone-only.npz")

for i, b_layer in enumerate(backbone.layers):
    layer = yolo.backbone[i]

    print(i)

    if isinstance(layer, Conv2D):
        assert cp.array_equal(layer.weights, b_layer.weights)
        if layer.bias is not None:
            assert cp.array_equal(layer.bias, b_layer.bias)
    elif isinstance(layer, BatchNorm2D):
        if layer.affine:
            assert cp.array_equal(layer.gamma, b_layer.gamma)
            assert cp.array_equal(layer.beta, b_layer.beta)
        print("Y", layer.running_mean)
        print("B", b_layer.running_mean)
        assert cp.array_equal(layer.running_mean, b_layer.running_mean)
        assert cp.array_equal(layer.running_var, b_layer.running_var)
    elif isinstance(layer, Linear):
        assert cp.array_equal(layer.W, b_layer.W)
        if layer.use_bias:
            assert cp.array_equal(layer.b, b_layer.b)


0
1
Y [-6.9290310e-02 -3.5094970e-01 -4.3100527e-01 -1.0393789e-02
  1.8781137e-02  3.5179475e-01  6.1029124e-01  3.6022383e-01
  3.3352160e-01  3.5508692e-01  7.1631931e-02 -6.3321374e-02
  8.6207002e-01  2.8959364e-02 -3.1676179e-01 -3.0345348e-01
 -4.5588630e-01 -4.5654505e-01 -1.6596492e-01 -2.6014814e-01
 -5.6572998e-01 -2.8661975e-01 -3.8916528e-01 -7.1339941e-01
  3.7284091e-01 -6.6125810e-02  3.1302460e-02  5.5476642e-01
  1.0396233e-02  1.6032391e-04 -9.9198833e-02 -1.7703128e-01
 -3.7271303e-01  6.5389611e-03  4.9293283e-01  4.4869611e-01
 -8.0341235e-02 -5.2009952e-01  9.2587337e-02 -4.3526158e-01
 -7.2467998e-02  2.7875575e-01  6.6696882e-01 -7.3371619e-02
  4.6949157e-01 -3.0424577e-01 -7.2743708e-01  2.0612775e-01
 -5.1916105e-01 -5.9665972e-01 -1.0524308e-01 -1.6036972e-01
 -1.0564118e-01 -5.6180125e-01 -3.5880557e-01  1.1175231e-02
 -6.3774467e-02 -1.9805558e-01 -1.1311037e-01  2.5764193e-02
  6.0127709e-02  6.6406548e-01 -1.6904461e-01 -3.4433910e-01]
B [-6.9290310e-02

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

LOCAL_CKPT_PATH = "/content/yolo-backbone-only.npz"
DRIVE_CKPT_PATH = "/content/drive/MyDrive/yolov1-cupy/yolo-backbone-only.npz"
!mkdir -p "/content/drive/MyDrive/yolov1-cupy"
shutil.copy2(LOCAL_CKPT_PATH, DRIVE_CKPT_PATH)
print("Saved locally to", LOCAL_CKPT_PATH)
print("Uploaded to", DRIVE_CKPT_PATH)


Mounted at /content/drive
Saved locally to /content/yolo-backbone-only.npz
Uploaded to /content/drive/MyDrive/yolov1-cupy/yolo-backbone-only.npz
